In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
!pip install -q transformers datasets accelerate evaluate scikit-learn


In [33]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [34]:
DATA_PATH = '/content/drive/MyDrive/Assamese_Sentiment_Analysis/Data/Assamese_Movie_Review_Cleaned_8987.csv'
MODEL_NAME = 'google/muril-base-cased'


In [35]:
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()


(8987, 3)


,ID,Reviews,Sentiment
0,0,বোকা নাটক,negative
1,1,নাটকখন বৰ ধুনীয়া আছিল,positive
2,2,এখন ভাল ঘৰ এনে হব লাগে,positive
3,3,এইটো আশ্চৰ্যকৰ আছিল,positive
4,4,মই বহুদিন অপেক্ষা কৰি আছো. অৱশেষত দেখিলো. কেৱল...,positive


In [36]:
df['Sentiment'].value_counts(dropna=False)


,count
Sentiment,
positive,5677
negative,3309
NaN,1


In [38]:
# Show the problematic sentiment entries
df.loc[df['Sentiment'].isna(), 'Sentiment'].value_counts(dropna=False)

,count
Sentiment,
NaN,1


In [40]:
df[df['Sentiment'].isna()]['Sentiment'].unique()


array([nan], dtype=object)

# Dataset Class

In [49]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Model Configuration and Metrics

In [50]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')

    return {
        'accuracy': accuracy,
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': f1
    }

OUTPUT_DIR = "./results"

In [41]:
# Remove corrupted row with header leakage
df = df[~((df['ID'] == 4227) & (df['Reviews'] == 'Reviews'))].reset_index(drop=True)

print("Dataset shape after removing corrupted row:", df.shape)


Dataset shape after removing corrupted row: (8986, 3)


In [42]:
assert not ((df['Reviews'] == 'Reviews') | (df['Sentiment'].isna())).any(), \
    "Corrupted rows still exist in the dataset"


In [43]:
df['Sentiment'] = (
    df['Sentiment']
    .astype(str)
    .str.strip()
    .str.lower()
)

label_map = {'negative': 0, 'positive': 1}
df['label'] = df['Sentiment'].map(label_map)

df = df.dropna(subset=['label']).reset_index(drop=True)

assert df['label'].isna().sum() == 0


In [44]:
# Ensure all reviews are strings (critical for tokenizers)
df['Reviews'] = df['Reviews'].astype(str)

# Remove empty or whitespace-only reviews
df = df[df['Reviews'].str.strip() != ""].reset_index(drop=True)

# Sanity checks
assert df['Reviews'].isna().sum() == 0, "NaN found in Reviews"
assert df['label'].isna().sum() == 0, "NaN found in labels"

print("Pre-tokenization normalization completed")
print("Dataset shape:", df.shape)


Pre-tokenization normalization completed
Dataset shape: (8986, 4)


# Add the seed function

In [45]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Define Seeds

In [46]:
SEEDS = [42, 7, 21, 99, 123]

all_results = []

# Train test split

In [52]:
for seed in SEEDS:

    print("\n" + "="*60)
    print(f"RUNNING SEED {seed}")
    print("="*60)

    set_seed(seed)

    # -----------------------------
    # Train Validation Split
    # -----------------------------

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['Reviews'].tolist(),
        df['label'].tolist(),
        test_size=0.2,
        random_state=seed,
        stratify=df['label']
    )

    # -----------------------------
    # Tokenization
    # -----------------------------

    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=128
    )
    val_encodings = tokenizer(
        val_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    train_dataset = SentimentDataset(
        train_encodings,
        train_labels
    )

    val_dataset = SentimentDataset(
        val_encodings,
        val_labels
    )

    # -----------------------------
    # Fresh Model
    # -----------------------------

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    # -----------------------------
    # Training Arguments
    # -----------------------------

    training_args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}_seed_{seed}",

        eval_strategy='epoch',
        save_strategy='epoch',

        learning_rate=2e-5,

        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,

        num_train_epochs=4,

        weight_decay=0.01,

        warmup_ratio=0.1,

        logging_steps=100,

        load_best_model_at_end=True,

        metric_for_best_model='macro_f1',

        greater_is_better=True,

        save_total_limit=2,

        report_to='none'
    )

    # -----------------------------
    # Trainer
    # -----------------------------

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # -----------------------------
    # Train
    # -----------------------------

    trainer.train()

    # -----------------------------
    # Evaluate
    # -----------------------------

    results = trainer.evaluate()

    all_results.append({
        "seed": seed,
        "accuracy": results["eval_accuracy"],
        "macro_f1": results["eval_macro_f1"],
        "macro_precision": results["eval_macro_precision"],
        "macro_recall": results["eval_macro_recall"]
    })


RUNNING SEED 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.275290,0.210158,0.945495,0.945020,0.937327,0.940927
2,0.186791,0.205438,0.943826,0.948708,0.930649,0.938459
3,0.099844,0.160223,0.960512,0.954912,0.961186,0.957860
4,0.087110,0.157969,0.959399,0.956763,0.955894,0.956325


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

# Create Results Table

In [53]:
results_df = pd.DataFrame(all_results)

results_df

,seed,accuracy,macro_f1,macro_precision,macro_recall
0,42,0.959956,0.957254,0.954422,0.960431
1,7,0.959399,0.956489,0.955252,0.957785
2,21,0.962180,0.959278,0.960172,0.958410
3,99,0.968298,0.966026,0.964762,0.967348
4,123,0.964405,0.961818,0.960971,0.962692


# Mean ± Standard Deviation

In [54]:
summary = pd.DataFrame({
    "Mean": results_df.mean(numeric_only=True),
    "Std": results_df.std(numeric_only=True)
})

summary

,Mean,Std
seed,58.400000,50.326931
accuracy,0.962848,0.003634
macro_f1,0.960173,0.003868
macro_precision,0.959116,0.004284
macro_recall,0.961333,0.003872
